## 합성곱 신경망

### 파이토치로 패션 MNIST CNN 실습
- 텐서플로(케라스)로 CNN, 파이토치로 CNN -> 아무 차이 없음

#### 데이터셋 불러오기

In [1]:
# 사용 모듈 로드
import torch
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader

In [4]:
# 이전에는 sklearn에 있는 StandardScaler 사용
# 정규화 및 텐서 변환
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, ), (0.5, ))  # Fashion-MNIST 자주 사용하는 정규화 범위
])

# Fasion-MNIST를 DataSet으로 변경
# 현재위치에 다운로드
# keras는 C:\Users\Admin\.keras\datasets
# PyTorch는 현재위치에 다운로드
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
val_dataset = datasets.FashionMNIST(root='./data', train=False, transform=transform)

# Dataset을 Dataloader로 변경

#### 클래스 레이블
|레이블|0|1|2|3|4|5|6|7|8|9|
|:---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
|패션MNIST|티셔츠|바지|스웨터|드레스|코드|샌달|셔츠|스니커즈|가방|앵클 부츠|

#### CNN 모델 정의

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)  # 2x2 최대 풀링 / 사이즈 반으로 줄이는 것
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)   # 28x28 -> 14x14 -> 7x7
        self.fc2 = nn.Linear(128, 10)   # 10개 클래스(0 ~ 9까지)
    
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))    # conv1 통과시키고 relu 활성화 후 MaxPool -> 14x14 특성맵
        x = self.pool(F.relu(self.conv2(x)))    # conv2 통과시키고 relu 활성화 후 MaxPool -> 7x7 특성맵
        x = x.view(-1, 64*7*7)                  # flattern - 1차원 배열화, 3136개 입력
        x = F.relu(self.fc1(x))                 # Dense layer 통과, relu 활성화
        x = F.softmax(self.fc2(x))              # F.softmax() deprecated됨
        return x
    
## keras CNN과 완전 일치

#### 훈련 / 평가 루프

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # 학습률 0.01 -> 0.001로 조정

In [7]:
# 훈련함수
def train(model, dataloader, criterion, optimizer):
    model.train()   # 훈련모드
    total_loss = 0

    for X_batch, y_batch in dataloader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)

In [8]:
# 평가함수
def evaluate(model, dataloader, criterion):
    model.eval()    # 검증모드
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item()

            preds = outputs.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)

    return total_loss / len(dataloader), correct / total

#### 훈련실행

In [ ]:
EPOCH = 10

for epoch in range(EPOCH):
    train_loss = train(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, val_loader, criterion)

    print(f'{epoch+1}/{EPOCH} Train Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f}, Val Accuracy: {val_acc:2%}')